In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping
where hcp_npi in ('1861578924', '1265183586',
'1154853661',
'1306574504',
'1790314763',
'1447684535')

In [0]:
CREATE OR replace temp VIEW mpsii_tx_claims AS 

SELECT a.*, b.PRIMARY_SPECIALTY as hcp_primary_specialty
from (SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2) and fill_date between '2020-08-01' AND '2025-07-31') as a
left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi

In [0]:
select * from mpsii_tx_claims where npi is not null

In [0]:
-- Getting all the treatment claims of patients with 2 E761 Dx claim + Elaprase Tx Claims

select hcp_primary_specialty, count(distinct patient_id)
from mpsii_tx_claims
where hcp_primary_specialty is not null
group by 1
order by 2 desc

In [0]:
-- Elaprase Treated patients (Can or cannot be mpsii diagnosed)
-- A single patient can be attributed to multiple HCPs

with elaprase_tx_claims as (SELECT a.*, b.PRIMARY_SPECIALTY as hcp_primary_specialty
from (SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where fill_date between '2020-08-01' AND '2025-07-31') as a
left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi)
select npi, count(distinct patient_id) as total_patients
from elaprase_tx_claims
where npi is not null
group by 1 order by 2 desc

In [0]:
-- Getting the first diagnosis claim of every patient and checking the npi tagged to that claim

with diagnosis_table as (SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES ilike '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE ILIKE '%E761%'
      AND TRANSACTION_STATUS = 'PAID'),
first_incidence AS (
  SELECT patient_id, min(fill_date) as first_incidence_date
FROM (
    select * from diagnosis_table
  ) AS combined
  group by patient_id
),
diagnosis_table_first_incidence_only as (
  select * from diagnosis_table
where (patient_id, fill_date) in (select distinct patient_id, first_incidence_date from first_incidence)
)
select npi, count(distinct patient_id) as diagnosed_patients
from diagnosis_table_first_incidence_only
where npi is not null
group by 1 order by 2 desc

In [0]:
-- Elaprase Treated patients (Only mpsii diagnosed patients)
-- A single patient can be attributed to multiple HCPs

select npi, count(distinct patient_id)
from mpsii_tx_claims
where npi is not null
group by 1 order by 2 desc

In [0]:
-- POS for all the elaprase treated patients (Only mpsii diagnosed patients)

with pos_patient_count as (
  select place_of_service, count(distinct patient_id) as total_patients
from mpsii_tx_claims
where place_of_service is not null
group by 1 order by 2 desc
)
select distinct pos.description, total_patients
from pos_patient_count as a
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  order by 2 desc

In [0]:
-- Getting all the tx claims, using komodo provider info and tagging primary hco npi and then using open data to get the hco type

with tx_claims_with_hco as (
  select a.*, b.HCO_PRIMARY_NPI as hco_npi, c.hco_type__v as hco_type
from mpsii_tx_claims as a
left join com_edp_prd.com_raw.kom_providers as b
on a.npi = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
left join com_edp_prd.com_raw.vod_hco as c
on b.HCO_PRIMARY_NPI = c.npi_num__v
),
hco_type_patient_count as (
  select a.hco_type, count(distinct a.patient_id) as total_patients
  from tx_claims_with_hco as a
  where hco_type is not null
  group by 1 order by 2 desc
)
select b.name as hco_type_info, a.total_patients
from hco_type_patient_count as a
left join com_edp_prd.com_raw.vod_references as b
on a.hco_type = b.code and b.reference_type = 'HCOType'
order by 2 desc

In [0]:
-- Tx claims with hco info, All the hco's in here have the infusion capability 
-- Assumption : The patient received treatment at the primary HCO location of the HCP associated with the patient’s treatment (Tx) claim.

with tx_claims_with_hco as (
  select a.*, b.HCO_PRIMARY_NPI as hco_npi
from mpsii_tx_claims as a
left join com_edp_prd.com_raw.kom_providers as b
on a.npi = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
)
select distinct hco_npi, 1 as has_infusion_capability
from tx_claims_with_hco

In [0]:
--  We are analyzing all medical claims for Elaprase and their associated procedures. Pharmacy claims are excluded since they only contain a prescriber NPI and not a rendering NPI. Our goal is to identify, for each rendering NPI, the number of distinct referring NPIs associated with it - essentially determining which HCPs receive the most referrals from other HCPs. The analysis is limited to the medical events table.

with tx_medical_claims as (
  SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS rendering_npi,
                REFERRING_NPI AS referring_npi,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS rendering_npi,
                REFERRING_NPI AS referring_npi,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2) and fill_date between '2020-08-01' AND '2025-07-31'
)
select rendering_npi, count(distinct referring_npi) as unique_referring_npi
from tx_medical_claims
where rendering_npi is not null
group by 1 order by 2 desc

In [0]:
--  The cohort is same as above when looking at the referral network strength. Just now we are looking at the referring npis and the distinct count of patients they have referred

with tx_medical_claims as (
  SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS rendering_npi,
                REFERRING_NPI AS referring_npi,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS rendering_npi,
                REFERRING_NPI AS referring_npi,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2) and fill_date between '2020-08-01' AND '2025-07-31'
)
select referring_npi, count(distinct patient_id) as total_referred_patients
from tx_medical_claims
where referring_npi is not null
group by 1 order by 2 desc